# July 2021 floods: starter notebook

This notebook does two things:
1. **Connects** you to the challenge data in the cloud. Nothing is downloaded, and no login is needed: the data is public.
2. **Shows what data is available** (EMS flood maps, EFAS flood hazard, GHS-OBAT buildings, GHS-POP population for DE, BE, NL).

Click **Run all**. Works in Colab and on a laptop.

<details>
<summary><b>Not using Colab? (VS Code, Jupyter, any Python)</b></summary>

1. `pip install gcsfs pyogrio rasterio geopandas shapely ipykernel`
   (on Windows, if that fails: `conda install -c conda-forge geopandas rasterio pyogrio gcsfs`)
2. VS Code: install the Python and Jupyter extensions and pick your environment as kernel. If the packages are already installed there, you can skip the `pip install` cell below. After installing, restart the kernel.

**Problems:** *connection or timeout errors* → check your internet connection and retry · *`No such file` on a path* → run the Explore cell to see the exact file names · *`ModuleNotFoundError`* → the kernel and your terminal use different Pythons: install with `%pip install gcsfs` inside the notebook, then restart the kernel.

</details>

## 1. Setup

In [ ]:
%%capture
%pip install -q gcsfs pyogrio rasterio geopandas shapely

In [ ]:
import os, re, warnings
from pathlib import Path
import pandas as pd
warnings.filterwarnings("ignore")

BUCKET = "datasets_challenge"
FOLDER = "working_data"

## 2. Connect

In [ ]:
import gcsfs

# The bucket is public: no login needed.
os.environ["GS_NO_SIGN_REQUEST"] = "YES"      # lets GDAL (rasterio, pyogrio) read it anonymously
GDAL_PREFIX = "/vsigs/"
ROOT = f"{BUCKET}/{FOLDER}"
fs = gcsfs.GCSFileSystem(token="anon")
print("Connected:", [Path(p).name for p in fs.ls(ROOT)])

## 3. Explore

In [ ]:
for folder in ["cems-efas", "ghs_pop", "obat_be", "obat_de", "obat_nl"]:
    print(f"\n{folder}")
    for p in sorted(fs.find(f"{ROOT}/{folder}")):
        if p.lower().endswith((".tif", ".tiff", ".gpkg")): print("  ", Path(p).name, f"{fs.size(p)/1e6:,.0f} MB")

# EMS: one row per area (aoi), type (DEL/GRA) and version, with its layers
EMS_RE = re.compile(r"(EMSR\d+)_(AOI\d+)_([A-Z]+)_(PRODUCT|MONIT\d*)_(\w+?)_r\d+_v\d+", re.I)
rows = []
for cc in ("de", "be", "nl"):
    for p in fs.find(f"{ROOT}/ems_{cc}"):
        m = EMS_RE.search(Path(p).name) if p.lower().endswith(".shp") else None
        if m: rows.append(dict(cc=cc, aoi=m[2], ptype=m[3].upper(), ver=m[4].upper(), layer=m[5]))
display(pd.DataFrame(rows).groupby(["cc", "aoi", "ptype", "ver"]).layer.apply(lambda s: ", ".join(sorted(s))).to_frame())

## 4. Your analysis
Read data straight from the bucket, no login needed: list files with `fs` (gcsfs), or open them with GDAL-based libraries using the `GDAL_PREFIX` path prefix, for example `rasterio.open(f"{GDAL_PREFIX}{path}")`. The bucket is read-only, so save results to the runtime disk and download them from the file browser.

In [ ]:
# your code